# Demo 1: Localizing the goal heads

**Insight.** A small set of attention heads, the goal heads, carries *which* sub-goal the LM is about to implement.
Copied at a single token from a run in another programming language, they make the LM implement the other sub-goal.

**Setup.** Three prompts that ask for a one-line predicate:
- **source**: predicate A (`is_palindrome`) in C++, stopped at `return `;
- **destination**: predicate B (`is_upper`) in Python, stopped at `return s == s`, a prefix that A and B share in
  Python and after which they diverge (`s == s[::-1]` vs `s == s.upper()`);
- **target**: predicate A in Python, stopped at the same prefix. It is never patched; it only shows what success looks like.

**Intervention.** At the last token, the goal heads' outputs in the destination run are replaced by the same heads'
outputs in the C++ source run. If the destination now continues like the target, the heads carried the goal: the
source is in another language and never saw the Python prefix, so only *which predicate to implement* can move across.
The goal heads were found by training a sparse mask over all 2,560 heads to make exactly this transfer work on ~1,400
such instances (the training is not run here).

**Paper.** §3.1, App. C (Figure 9).

Needs one 80 GB GPU (Qwen2.5-Coder-32B-Instruct in bf16). The executed outputs are saved in this file.

**The setup** (schematics from the lab talk):

![Localizing goal heads: transplant every head's output from the C++ source run into the Python destination run at the pivot, and learn which heads carry the sub-goal (§3.1, App. C).](figures/01_localize_goal_heads.png)

*Localizing goal heads: transplant every head's output from the C++ source run into the Python destination run at the pivot, and learn which heads carry the sub-goal (§3.1, App. C).*

In [1]:
# repo root = first parent dir that contains src/
import os, sys
_d = os.getcwd()
while _d != '/' and not os.path.isdir(os.path.join(_d, 'src')):
    _d = os.path.dirname(_d)
sys.path.insert(0, _d); os.chdir(_d)
print('repo root:', _d)

# no progress bars: keeps tqdm / HF widget outputs out of the saved notebook
os.environ['TQDM_DISABLE'] = '1'
import functools, tqdm.std, tqdm.notebook, transformers, huggingface_hub
for _c in (tqdm.std.tqdm, tqdm.notebook.tqdm):   # tqdm.notebook ignores TQDM_DISABLE, so force disable=True
    _c.__init__ = functools.partialmethod(_c.__init__, disable=True)
transformers.utils.logging.disable_progress_bar()
transformers.utils.logging.set_verbosity_error()      # no generation-config warnings
huggingface_hub.utils.disable_progress_bars()

# as in the paper's figures: source red, destination green, patched run purple
from typing import Literal
ide_theme: Literal['dark', 'light'] = 'dark'
if ide_theme == 'dark':   # light shades; the task-span fill reuses the text color
    SRC_C, DST_C, PAT_C = 'salmon1', 'sea_green3', 'plum2'
    SRC_BG, DST_BG      = SRC_C, DST_C
    PIVOT_BG            = 'bright_yellow'
else:                     # the paper's colors (Figure 5)
    SRC_C, DST_C, PAT_C = '#cc0000', '#38761d', '#5b277d'
    SRC_BG, DST_BG      = '#f4cccc', '#d9ead3'
    PIVOT_BG            = '#ffe6b2'

repo root: /disk/u/arnab/Codes/Projects/goals


In [2]:
CUDA         = '7'
MODEL        = 'Qwen/Qwen2.5-Coder-32B-Instruct'
HEAD_SET     = 'head_sets/Qwen2.5-Coder-32B-Instruct/goal__lam20_thr0.9_band25-56.json'     # the 122 goal heads
CONTROL_SET  = 'head_sets/Qwen2.5-Coder-32B-Instruct/control/control122__laterhalf.json'     # 122 random heads, layers >= 32
SOURCE_PRED  = 'is_palindrome'   # predicate A, asked in C++ (source)
DEST_PRED    = 'is_upper'        # predicate B, asked in Python (destination)
PIVOT_PREFIX = 's == s'          # the shared Python code after `return `
N_SWEEP      = 60                # pivot instances for the check in section 4
SEED         = 0

os.environ['CUDA_VISIBLE_DEVICES'] = CUDA

## Load the model
Qwen2.5-Coder-32B-Instruct, the 122 goal heads, and two random sets of 122 heads (none of them goal heads) as controls.

In [3]:
import random, torch
import numpy as np
from rich.console import Console
from rich.table import Table
from rich.markup import escape
from rich.text import Text

from src.models import ModelandTokenizer
from src.utils import experiment_utils
from src.goals.head_sets import load_json_heads
from src.goals.head_loc_data import HeadLocDataset
from src.goals.head_loc_pivot import build_cpp_prompt, build_python_prompt, get_next_token_probs_batch
from src.goals.head_loc_attn import HeadMask, get_o_proj_inputs_batch, forward_with_src_replacement
from src.tokens import prepare_input

console = Console(width=130, highlight=False)
torch.set_grad_enabled(False)
experiment_utils.set_seed(SEED)

mt = ModelandTokenizer(model_key=MODEL, dtype=torch.bfloat16, device_map={'': 0}, attn_implementation='eager')
goal_heads = load_json_heads(HEAD_SET, mt.n_layer, mt.n_heads)
control_heads = load_json_heads(CONTROL_SET, mt.n_layer, mt.n_heads)
# a second control: 122 random non-goal heads from the goal heads' own layers (28-55)
_rng = random.Random(SEED)
_gs = set(goal_heads)
_pool = [(l, h) for l in range(28, 56) for h in range(mt.n_heads) if (l, h) not in _gs]
band_random_heads = sorted(_rng.sample(_pool, len(goal_heads)))
print(f'{MODEL}: {mt.n_layer} layers x {mt.n_heads} heads (head_dim {mt.head_dim})')
print(f'goal heads: {len(goal_heads)} | random (layers >= 32): {len(control_heads)} | random (layers 28-55): {len(band_random_heads)}')

Qwen/Qwen2.5-Coder-32B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory: /disk/u/models


Qwen/Qwen2.5-Coder-32B-Instruct: 64 layers x 40 heads (head_dim 128)
goal heads: 122 | random (layers >= 32): 122 | random (layers 28-55): 122


## 1. A pivot instance
The three prompts (only the code part the LM is continuing). The highlighted tail is where the prompt stops: the
patch acts on its last token.

In [4]:
ds = HeadLocDataset.load(model_name=MODEL.split('/')[-1])
print(f'{len(ds.samples)} predicates | {len(ds.pairs)} verified predicate pairs | {len(ds.pivot_instances)} pivot instances')

pair, pi = next((p, q) for p, q in ds.pivot_instances
                if p.source_predicate == SOURCE_PRED and p.destination_predicate == DEST_PRED and q.pivot_prefix == PIVOT_PREFIX)
src_s, dst_s = ds.get_sample(pair.source_predicate), ds.get_sample(pair.destination_predicate)
src_prompt  = build_cpp_prompt(mt, src_s, include_docstring=True)
dest_prompt = build_python_prompt(mt, dst_s, pivot_prefix=pi.pivot_prefix, include_docstring=True)
targ_prompt = build_python_prompt(mt, src_s, pivot_prefix=pi.pivot_prefix, include_docstring=True)

def assistant_part(p):  # the code the model is continuing (drop system/user turns for display)
    return p.split('<|im_start|>assistant\n')[-1]

def show(p, title, color):
    code = assistant_part(p)
    k = code.rindex('return')                         # the prompt stops at `return ` (+ the shared prefix)
    txt = Text(code[:k], style=color); txt.append(code[k:], style=f'bold black on {PIVOT_BG}')
    console.rule(f'[bold {color}]{title}[/]'); console.print(txt)

show(src_prompt,  'source: A in C++', SRC_C)
show(dest_prompt, 'destination: B in Python', DST_C)
show(targ_prompt, 'target: A in Python (reference, never patched)', PAT_C)
console.rule()

21 predicates | 110 verified predicate pairs | 1613 pivot instances


──────────────────────────────────────────────────────── source: A in C++ ────────────────────────────────────────────────────────

<code>
// Return True if s reads the same forwards and backwards.
bool isPalindrome(string s) { return 

──────────────────────────────────────────────────── destination: B in Python ────────────────────────────────────────────────────

<code>
def is_upper(s: str) -> bool:
    """Return True if all alphabetic characters in s are uppercase."""
    return s == s

───────────────────────────────────────── target: A in Python (reference, never patched) ─────────────────────────────────────────

<code>
def is_palindrome(s: str) -> bool:
    """Return True if s reads the same forwards and backwards."""
    return s == s

──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

## 2. Destination vs. target: where the two predicates diverge
At the pivot the destination and target prompts have the same Python prefix and differ only in the predicate requested.
Their next-token distributions should be nearly disjoint.

In [5]:
P = get_next_token_probs_batch(mt, [dest_prompt, targ_prompt])   # (2, vocab)
dest_id, targ_id = P[0].argmax().item(), P[1].argmax().item()
tok = lambda i: repr(mt.tokenizer.decode([i]))

def topk_str(p, k=5):
    v, i = p.topk(k)
    return '  '.join(f'{tok(j)} {x:.2f}' for x, j in zip(v.tolist(), i.tolist()))

t = Table(title='next-token distribution at the pivot (clean runs)')
t.add_column('run'); t.add_column('top-5')
t.add_row(f'[{DST_C}]destination (B)[/]', escape(topk_str(P[0]))); t.add_row(f'[{PAT_C}]target (A)[/]', escape(topk_str(P[1])))
console.print(t)
print(f'dest top {tok(dest_id)} | target top {tok(targ_id)} | p_dest(target tok) = {P[0, targ_id]:.4f}')

                         next-token distribution at the pivot (clean runs)                          
┏━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ run             ┃ top-5                                                                          ┃
┡━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ destination (B) │ '.upper' 1.00  '.toUpperCase' 0.00  ' upper' 0.00  '_upper' 0.00  'upper' 0.00 │
│ target (A)      │ '[::-' 1.00  '[' 0.00  '[-' 0.00  '[:' 0.00  '.reverse' 0.00                   │
└─────────────────┴────────────────────────────────────────────────────────────────────────────────┘

dest top '.upper' | target top '[::-' | p_dest(target tok) = 0.0000


## 3. The intervention: transplant the goal heads from C++ into Python
For each of the 122 goal heads, its output at the destination's last token is replaced by the same head's output at
the last token of the C++ source; every other head is left alone. Look at whether the destination's top next token
moves to the target's.

In [6]:
LAYERS = list(range(mt.n_layer))

def binary_mask(heads):
    m = HeadMask(n_layers_tested=mt.n_layer, n_heads=mt.n_heads)
    m.values.data.zero_()
    for l, h in heads:
        m.values.data[l, h] = 1.0
    return m

def patched_probs(src_prompt, dest_prompt, heads):
    so = get_o_proj_inputs_batch(mt, [src_prompt], LAYERS)            # {layer: (1, n_heads*head_dim)} at the last token
    src_vals = torch.stack([so[l][0] for l in LAYERS])               # (n_layers, n_heads*head_dim)
    logits = forward_with_src_replacement(mt, prepare_input([dest_prompt], tokenizer=mt), src_vals, binary_mask(heads), LAYERS)
    return torch.softmax(logits.float(), -1).cpu()

p_goal = patched_probs(src_prompt, dest_prompt, goal_heads)

t = Table(title='before / after patching the 122 goal heads', show_lines=True)
t.add_column('run'); t.add_column('top-1'); t.add_column(f'p({tok(targ_id)}) target', justify='right')
t.add_column(f'p({tok(dest_id)}) dest', justify='right'); t.add_column('top-5')
for name, p in [(f'[{DST_C}]destination, clean[/]', P[0]), (f'[bold {PAT_C}]destination + goal heads from C++[/]', p_goal),
                (f'[{PAT_C}]target, clean (reference)[/]', P[1])]:
    t.add_row(name, tok(p.argmax().item()), f'{p[targ_id]:.3f}', f'{p[dest_id]:.3f}', escape(topk_str(p)))
console.print(t)

                                            before / after patching the 122 goal heads                                            
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ run                               ┃ top-1    ┃ p('[::-') target ┃ p('.upper') dest ┃ top-5                                     ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ destination, clean                │ '.upper' │            0.000 │            1.000 │ '.upper' 1.00  '.toUpperCase' 0.00  '     │
│                                   │          │                  │                  │ upper' 0.00  '_upper' 0.00  'upper' 0.00  │
├───────────────────────────────────┼──────────┼──────────────────┼──────────────────┼───────────────────────────────────────────┤
│ destination + goal heads from C++ │ '[::-'   │            1.000 │            0.000 │ '[::-' 1.00  '.lower' 0.00  '.upper' 0.00 │
│                                   │          │                  │                  │ '[' 0.00  '[:' 0.00                       │
├───────────────────────────────────┼──────────┼──────────────────┼──────────────────┼───────────────────────────────────────────┤
│ target, clean (reference)         │ '[::-'   │            1.000 │            0.000 │ '[::-' 1.00  '[' 0.00  '[-' 0.00  '[:'    │
│                                   │          │                  │                  │ 0.00  '.reverse' 0.00                     │
└───────────────────────────────────┴──────────┴──────────────────┴──────────────────┴───────────────────────────────────────────┘

## 4. Control: is it just "patch any 122 heads"?
The same patch on the same example, through two random sets of 122 heads: one from the later half of the model, one
from the goal heads' own layers (28–55).

In [7]:
p_ctrl = patched_probs(src_prompt, dest_prompt, control_heads)
p_rand = patched_probs(src_prompt, dest_prompt, band_random_heads)
t = Table(title='same example, different head sets', show_lines=True)
t.add_column('run'); t.add_column('top-1'); t.add_column(f'p({tok(targ_id)}) target', justify='right')
t.add_column(f'p({tok(dest_id)}) dest', justify='right'); t.add_column('top-5')
for name, p in [(f'[{DST_C}]destination, clean[/]', P[0]), (f'[bold {PAT_C}]+ 122 goal heads[/]', p_goal),
                ('+ 122 random heads, layers >= 32', p_ctrl), ('+ 122 random heads, layers 28-55', p_rand),
                (f'[{PAT_C}]target, clean (reference)[/]', P[1])]:
    t.add_row(name, tok(p.argmax().item()), f'{p[targ_id]:.3f}', f'{p[dest_id]:.3f}', escape(topk_str(p)))
console.print(t)

                                                same example, different head sets                                                 
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ run                              ┃ top-1    ┃ p('[::-') target ┃ p('.upper') dest ┃ top-5                                      ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ destination, clean               │ '.upper' │            0.000 │            1.000 │ '.upper' 1.00  '.toUpperCase' 0.00  '      │
│                                  │          │                  │                  │ upper' 0.00  '_upper' 0.00  'upper' 0.00   │
├──────────────────────────────────┼──────────┼──────────────────┼──────────────────┼────────────────────────────────────────────┤
│ + 122 goal heads                 │ '[::-'   │            1.000 │            0.000 │ '[::-' 1.00  '.lower' 0.00  '.upper' 0.00  │
│                                  │          │                  │                  │ '[' 0.00  '[:' 0.00                        │
├──────────────────────────────────┼──────────┼──────────────────┼──────────────────┼────────────────────────────────────────────┤
│ + 122 random heads, layers >= 32 │ '.upper' │            0.000 │            1.000 │ '.upper' 1.00  '.toUpperCase' 0.00  '      │
│                                  │          │                  │                  │ upper' 0.00  '_upper' 0.00  'upper' 0.00   │
├──────────────────────────────────┼──────────┼──────────────────┼──────────────────┼────────────────────────────────────────────┤
│ + 122 random heads, layers 28-55 │ '.upper' │            0.000 │            1.000 │ '.upper' 1.00  '.toUpperCase' 0.00  '      │
│                                  │          │                  │                  │ upper' 0.00  '_upper' 0.00  'upper' 0.00   │
├──────────────────────────────────┼──────────┼──────────────────┼──────────────────┼────────────────────────────────────────────┤
│ target, clean (reference)        │ '[::-'   │            1.000 │            0.000 │ '[::-' 1.00  '[' 0.00  '[-' 0.00  '[:'     │
│                                  │          │                  │                  │ 0.00  '.reverse' 0.00                      │
└──────────────────────────────────┴──────────┴──────────────────┴──────────────────┴────────────────────────────────────────────┘

### Beyond one example
One instance can be lucky. Below, the same three head sets on an evenly spaced sample of pivot instances. Success =
the patched top-1 token equals the target's top-1. Most of these instances were used to train the mask, so this is
a sanity check of the head set, not a held-out score.

In [8]:
pis = ds.pivot_instances
sample = [pis[i] for i in np.linspace(0, len(pis) - 1, N_SWEEP).astype(int)]
hits = {'122 goal heads': 0, '122 random heads, layers >= 32': 0, '122 random heads, layers 28-55': 0}
for pr, q in sample:
    a, b = ds.get_sample(pr.source_predicate), ds.get_sample(pr.destination_predicate)
    sp = build_cpp_prompt(mt, a, include_docstring=True)
    dp = build_python_prompt(mt, b, pivot_prefix=q.pivot_prefix, include_docstring=True)
    tp = build_python_prompt(mt, a, pivot_prefix=q.pivot_prefix, include_docstring=True)
    tgt = get_next_token_probs_batch(mt, [tp])[0].argmax().item()
    for k, hs in zip(hits, (goal_heads, control_heads, band_random_heads)):
        hits[k] += int(patched_probs(sp, dp, hs).argmax().item() == tgt)
t = Table(title=f'patched top-1 == target top-1 on {len(sample)} pivot instances')
t.add_column('patched heads'); t.add_column('success', justify='right')
for k, v in hits.items(): t.add_row(k, f'{v}/{len(sample)} = {v / len(sample):.0%}')
console.print(t)

   patched top-1 == target top-1 on 60 pivot    
                   instances                    
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┓
┃ patched heads                  ┃     success ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━┩
│ 122 goal heads                 │ 50/60 = 83% │
│ 122 random heads, layers >= 32 │   0/60 = 0% │
│ 122 random heads, layers 28-55 │   2/60 = 3% │
└────────────────────────────────┴─────────────┘

## What to expect
<img src="figures/01_head_localization_results.png" width="400" alt="The 122 goal heads on the layer x head grid of Qwen2.5-Coder-32B-Instruct (Figure 9, right, in the paper)">

With the goal heads patched in from the C++ source, the destination's top next token should switch from its own
continuation to the target's, while 122 random heads leave it where it was. Across the pivot instances, the trained
mask keeps 122 of the model's 2,560 heads, all in layers 28–55 of the 64. See §3.1 in the paper; App. C (Figure 9)
shows this setup and the map of the selected heads.